# 0915 — SDS-PAGE annotation and analysis: two expression-test gels

Annotates the two 2026-09-15 Coomassie gels and asks whether any construct expressed.

## Verdict: inconclusive — no detectable induction on either gel

| | |
|---|---|
| **CDS11_WT** (lanes 2–5, gel 2) | lanes essentially **blank** — no discrete bands at all. Loading/lysis failure, not a negative result. |
| **CDS12_WT** (lanes 6–9, gel 2) | well loaded, rich *E. coli* lysate pattern, but **uninduced ≈ induced**. No 41.8 kDa band attributable to the construct. |
| **ANC_Node40** (lanes 2–5, gel 1) | **blank**. |
| **ANC_Node281** (lanes 6–9, gel 1) | faint bands, but they sit at **45–58 kDa, above** the predicted 42.3 kDa, and appear in the uninduced lanes too. |

**Ladder: Thermo PageRuler Prestained Protein Ladder, 170–10 kDa — confirmed against the product
sheet on 2026-09-15** (70 kDa orange, 10 kDa green). Calibration good to ±9 %, i.e. ±4 kDa on a
42 kDa target. Figures number lanes 1–10 by position in the loaded series; the label sheet numbers
the same wells 2–11 (§5).

Three independent reasons the gels cannot answer the question:

1. **Uninduced ≈ induced in every pair.** Band contrast at the predicted mass: CDS12 non 0.018/0.018 vs IPTG 0.013/0.039 — the IPTG-raw lane is *lower* than both uninduced lanes. e281 non 0.018/0.022 vs IPTG 0.027/0.005. These differences are the size of the background ripple.
2. **Whole-cell lysate.** *E. coli* has abundant endogenous bands right through 35–60 kDa, which is exactly where a 42 kDa product must be seen. With 15–26 detectable peaks per lane, "a band near 42 kDa" occurs by chance.
3. **No empty-vector or untransformed-BL21 control lane.** Without one, a band present in both uninduced and induced lanes cannot be assigned to the construct rather than to the host. This matters because pET-28a/BL21(DE3) is leaky — a real product *can* appear in an uninduced lane, so "present in both" is not by itself disproof.

## What to run next

**Ni-NTA pulldown before the gel, or an anti-His western.** All six constructs carry pET-28a's N-terminal His6 (verified in
[`0910_construct_protein_mw.ipynb`](0910_construct_protein_mw.ipynb) — the two "NOTAG" names are wrong), so both work
without new cloning, and both remove the lysate background that is defeating this gel. Add an untransformed-BL21 lane.

Worth re-checking first: the blank lanes. Four of eight sample lanes on gel 2 and gel 1 have essentially nothing in them, which points at the lysis or loading step rather than at expression.

## Inputs

| File | What |
|---|---|
| `data/assays/0915_sds_page/label.jpeg` | handwritten loading plan (transcribed below) |
| `data/assays/0915_sds_page/cds11-cds12.jpeg` | gel 2 — CDS11_WT / CDS12_WT |
| `data/assays/0915_sds_page/e40-e281.jpeg` | gel 1 — CDS12_ANC_Node40 / Node281 |

Both photos are phone shots of the gel on a white transilluminator: heavy vignette,
washed-out midtones, no sample bands visible without processing.

Predicted masses come from `0910_construct_protein_mw.ipynb`.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageOps
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.signal import find_peaks

ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
IMG  = ROOT/"data/assays/0915_sds_page"
OUT  = ROOT/"code/output/0915_gel_annotation"; OUT.mkdir(parents=True, exist_ok=True)

# Predicted masses (0910 notebook): includes pET-28a's 21 aa MGSSHHHHHHSSGLVPRGSHM leader
PREDICTED = {"CDS11": 72.02, "CDS12": 41.84, "e40": 42.20, "e281": 42.27}

def load(fn):
    "Phone photos carry an EXIF rotation; without transpose the gel comes out landscape."
    return np.asarray(ImageOps.exif_transpose(Image.open(IMG/fn)).convert("RGB")).astype(float)/255.

## 1. Loading plan, transcribed from `label.jpeg`

Both gels: 10 lanes, prestained ladder (`P.L`) in lanes 1 and 10, eight samples between.
Each construct occupies four lanes — uninduced/induced × undiluted/1:4.

| Lane | Gel 1 (`e40-e281`) | Gel 2 (`cds11-cds12`) |
|---|---|---|
| 1 | P.L | P.L |
| 2–5 | e40: non·raw, non·1:4, IPTG·raw, IPTG·1:4 | CDS11: non·raw, non·1:4, IPTG·raw, IPTG·1:4 |
| 6–9 | e281: non·raw, non·1:4, IPTG·raw, IPTG·1:4 | CDS12: non·raw, non·1:4, IPTG·raw, IPTG·1:4 |
| 10 | P.L | P.L |

The label also records four concentrations, presumably of the undiluted lysates:
`CDS11 0.126`, `e281 0.128`, `CDS12 0.109`, `e40 0.147` (units not written).

> ⚠️ **Two caveats.**
> 1. The label's `1:4` for the gel 2 `CDS12 IPTG` block is overwritten and was read by analogy with
>    the other three blocks. Lane numbering on the sheet disagrees with the gel — see §5.
> 2. **Left–right orientation is assumed, not proven.** Both outer lanes are ladder, so the images
>    carry nothing that fixes the reading direction; if a gel was flipped when it went on the
>    transilluminator, the CDS11 and CDS12 blocks swap. This does not change the "no induction"
>    verdict, but it does decide *which* construct's lanes are the blank ones.

## 2. Flat-fielding

Bands are invisible in the raw photos. Dividing by a large-sigma Gaussian estimate of the
illumination removes the transilluminator glow. This is a **non-uniform** correction, so every
figure below is produced twice and labelled:

- `_uniform` — a single linear contrast stretch, nothing else. Publication-safe under iGEM's
  image rules; use this one on the wiki.
- `_flattened` — background-divided, for visibility, disclosed in the caption.

All measurements below use the flattened image. No local, per-lane or per-band edits anywhere.
A colour variant of each is described in §7.

In [2]:
# gel body crops (the bright plateau inside the dark surround)
BOX = {"cds11-cds12.jpeg": (839, 3492, 1406, 4081),
       "e40-e281.jpeg":    (631, 3574, 1128, 4135)}

def flatfield(fn, sigma=70):
    x0,x1,y0,y1 = BOX[fn]
    lum = load(fn)[y0:y1, x0:x1].mean(2)
    return lum/np.clip(gaussian_filter(lum, sigma), 1e-3, None)   # ~1 = clean gel, <1 = band

def uniform(fn):
    x0,x1,y0,y1 = BOX[fn]
    return load(fn)[y0:y1, x0:x1].mean(2)

## 3. Which ladder is this?

Nothing in the repo records it (`lab_notebook/digitalized/` only runs to 07-16), so it has to come
from the image first and be checked against the tube afterwards. The prestained reference bands are
the giveaway: find the band whose red channel exceeds the mean of green and blue, and the band where
green does.

All four ladder lanes across the two gels give the same layout: **10 bands, the 4th from the top
strongly red, the lowest one green.**

In [3]:
LADDER_X = {"cds11-cds12.jpeg": (196, 2496), "e40-e281.jpeg": (225, 2760)}

def ladder_colours(fn, w=45):
    x0,x1,y0,y1 = BOX[fn]; g = load(fn)[y0:y1, x0:x1]
    rows = []
    for tag, lx in zip("LR", LADDER_X[fn]):
        strip = g[:, lx-w:lx+w, :].mean(1)                       # (H, 3)
        lum   = strip.mean(1)
        dark  = np.clip(gaussian_filter1d(lum, 60) - gaussian_filter1d(lum, 6), 0, None)
        bp, _ = find_peaks(dark, prominence=dark.max()*0.06, distance=22)
        R, G, B = strip[:,0], strip[:,1], strip[:,2]
        for i, p in enumerate(bp, 1):
            rg, gb = R[p]-(G[p]+B[p])/2, G[p]-(R[p]+B[p])/2
            rows.append(dict(gel=fn[:9], lane=tag, band=i, y=int(p),
                             colour="RED" if rg > .012 else ("GREEN" if gb > .006 else "blue"),
                             r_minus_gb=round(rg,3), g_minus_rb=round(gb,3)))
    return pd.DataFrame(rows)

cols = pd.concat([ladder_colours(f) for f in BOX])
print(cols[cols.colour != "blue"].to_string(index=False))

      gel lane  band    y colour  r_minus_gb  g_minus_rb
cds11-cds    L     4 1185    RED       0.227      -0.222
cds11-cds    L    10 2559  GREEN      -0.253       0.018
cds11-cds    R     4 1215    RED       0.277      -0.175
cds11-cds    R    10 2555  GREEN      -0.072       0.039
e40-e281.    L     4  998    RED       0.278      -0.182
e40-e281.    L    10 2501  GREEN      -0.220       0.055
e40-e281.    R     4  991    RED       0.189      -0.199


That layout is **Thermo PageRuler Prestained Protein Ladder** — 170, 130, 100, **70 (orange)**, 55,
40, 35, 25, 15, **10 (green)** kDa. **Confirmed against the product sheet on 2026-09-15.**

> A first pass read the top band as 180 kDa. The product sheet gives **170**. The correction is
> almost cosmetic — it moves the predicted-mass guides by under 3 px (< 0.1 % in mass) — but the
> 170 kDa band now fits its own calibration at 1.00x instead of 0.98x, which is a small independent
> confirmation that the assignment is right.

In [4]:
LADDER_MW = np.array([170,130,100,70,55,40,35,25,15,10], float)

# Band y-positions read off the profiles above and checked against the colour table.
# The two ladder lanes disagree slightly (gel smile/tilt); `tilt` is the mean L->R offset.
CAL = {"cds11-cds12.jpeg": dict(yL=[598,707,957,1184,1461,1744,1996,2172,2448,2559], tilt=+21.3),
       "e40-e281.jpeg":    dict(yL=[349,466,739,1001,1316,1640,1926,2099,2382,2501], tilt= -4.8)}

## 4. Calibration and its precision

`log10(MW)` against migration distance, cubic. The check that matters is not R² on one lane but
whether the **residuals repeat across independent ladder lanes** — a misassigned ladder would not
produce the same error pattern three times.

In [5]:
LAD_OBS = {   # every ladder lane with all 10 bands resolved
 "gel2 left":  [598,707,957,1184,1461,1744,1996,2172,2448,2559],
 "gel2 right": [642,740,987,1215,1474,1767,2020,2187,2451,2556],
 "gel1 left":  [349,466,739,1001,1316,1640,1926,2099,2382,2501],
}
rows=[]
for name, y in LAD_OBS.items():
    y = np.array(y, float); c3 = np.polyfit(y, np.log10(LADDER_MW), 3)
    fit = 10**np.polyval(c3, y)
    r2  = 1 - (np.log10(LADDER_MW)-np.polyval(c3,y)).var()/np.log10(LADDER_MW).var()
    rows.append(dict(lane=name, R2=round(r2,5), max_err=f"{np.abs(fit/LADDER_MW-1).max()*100:.0f}%",
                     **{f"{m:.0f}": round(f/m, 2) for m, f in zip(LADDER_MW, fit)}))
print(pd.DataFrame(rows).to_string(index=False))
print("\nThe per-band error pattern is identical across all three lanes -> the assignment is"
      "\nself-consistent and the residuals are a property of this ladder/gel, not noise."
      "\nWorking precision: about +/-9%, i.e. +/-4 kDa on a 42 kDa target.")

      lane      R2 max_err  170  130  100   70   55   40   35   25   15   10
 gel2 left 0.99655      9% 1.00 1.05 0.92 1.02 1.01 1.09 0.94 1.01 0.95 1.05
gel2 right 0.99618      9% 0.99 1.05 0.92 1.01 1.02 1.09 0.93 1.01 0.95 1.05
 gel1 left 0.99629      9% 1.00 1.05 0.92 1.01 1.02 1.09 0.92 1.01 0.95 1.05

The per-band error pattern is identical across all three lanes -> the assignment is
self-consistent and the residuals are a property of this ladder/gel, not noise.
Working precision: about +/-9%, i.e. +/-4 kDa on a 42 kDa target.


## 5. Lane grid, and a numbering discrepancy

The two ladder lanes fix both ends, and the eight sample lanes sit evenly spaced between them. The
fit is checkable, and it checks out: the predicted centres land on the four loaded lanes of each gel
to within a pixel.

**But the label sheet and the gel disagree about lane numbers.** On the sheet, `P.L` is written under
the number **2**, the four `CDS11` labels under **3–6**, `CDS12` under **7–10** and the second `P.L`
under **11** — i.e. a 12-well comb with wells 1 and 12 left empty. The gel says otherwise: there is
**less than one well-spacing of gel outside each ladder lane** (196 px on the left, 157 px on the
right, against a 256 px lane pitch), so no unused well can fit beyond them — the ladders are in the
outermost loaded wells.

Most likely the number row was written out as a generic 1–12 ruler before loading and the sample
names were then written slightly offset beneath it; the crossed-out `12` on the gel 1 sheet supports
that. Either way **the construct-to-lane mapping is unaffected** — ladder, then 4 x first construct,
then 4 x second construct, then ladder, which is exactly what the gel shows. Only the absolute well
number is in doubt.

The figures therefore number lanes **1–10 by position in the loaded series**, and say so, with the
sheet's 2–11 noted in the subtitle. Worth confirming the comb size (10-well vs 12-well) for the
notebook record.

In [6]:
def lane_centres(fn):
    xL, xR = LADDER_X[fn]
    return np.linspace(xL, xR, 10), xL, xR

for fn in BOX:
    lx, xL, xR = lane_centres(fn)
    print(f"{fn:20s} lanes {xL}..{xR}, spacing {(xR-xL)/9:6.1f} px -> "
          f"{', '.join(f'{v:.0f}' for v in lx)}")

cds11-cds12.jpeg     lanes 196..2496, spacing  255.6 px -> 196, 452, 707, 963, 1218, 1474, 1729, 1985, 2240, 2496
e40-e281.jpeg        lanes 225..2760, spacing  281.7 px -> 225, 507, 788, 1070, 1352, 1633, 1915, 2197, 2478, 2760


## 6. Quantification

Two numbers per lane:

- **total signal** — integrated darkness over the migration zone. This is *total protein loaded*, nothing more.
- **band contrast at the predicted mass** — the height of any local peak inside a ±8 % window
  around the predicted mass, above the median of the flanking gel. This is the one that would show induction.

In [7]:
LANES = {
 "cds11-cds12.jpeg": ["P.L","CDS11 non raw","CDS11 non 1:4","CDS11 IPTG raw","CDS11 IPTG 1:4",
                      "CDS12 non raw","CDS12 non 1:4","CDS12 IPTG raw","CDS12 IPTG 1:4","P.L"],
 "e40-e281.jpeg":    ["P.L","e40 non raw","e40 non 1:4","e40 IPTG raw","e40 IPTG 1:4",
                      "e281 non raw","e281 non 1:4","e281 IPTG raw","e281 IPTG 1:4","P.L"],
}
YFLOOR = {"cds11-cds12.jpeg": 0.90, "e40-e281.jpeg": 0.86}   # crop above the dye front

def analyse(fn):
    ff = flatfield(fn); H, W = ff.shape
    dark = np.clip(1-ff, 0, None)
    lx, xL, xR = lane_centres(fn); half = (xR-xL)/9/2
    yL = np.array(CAL[fn]["yL"], float); tilt = CAL[fn]["tilt"]
    zone = slice(int(0.04*H), int(YFLOOR[fn]*H))
    y_of = lambda mw, f: float(np.polyval(np.polyfit(np.log10(LADDER_MW), yL+f*tilt, 3), np.log10(mw)))

    rows = []
    for i, (x, nm) in enumerate(zip(lx, LANES[fn]), 1):
        f   = (x-xL)/(xR-xL)
        col = gaussian_filter1d(dark[:, int(x-half*.55):int(x+half*.55)].mean(1), 5)
        rec = dict(lane=i, sample=nm, total=round(float(col[zone].sum()), 1))
        if nm != "P.L":
            mw = PREDICTED[nm.split()[0]]
            ye = y_of(mw, f); wd = abs(y_of(mw*1.08, f) - y_of(mw*0.92, f))/2
            a, b = int(ye-wd), int(ye+wd)
            flank = np.r_[col[max(0,a-3*int(wd)):a], col[b:b+3*int(wd)]]
            rec |= dict(predicted_kDa=mw, y_predicted=int(ye),
                        contrast=round(float(col[a:b].max()-np.median(flank)), 4))
        rows.append(rec)
    df = pd.DataFrame(rows)
    ref = df.loc[df.sample_.isin(["P.L"]) if False else df["sample"].eq("P.L"), "total"].mean()
    df["total_vs_ladder"] = (df["total"]/ref).round(2)
    return df

for fn in BOX:
    print(f"\n=== {fn} ===")
    print(analyse(fn).to_string(index=False))


=== cds11-cds12.jpeg ===


 lane         sample  total  predicted_kDa  y_predicted  contrast  total_vs_ladder
    1            P.L   55.2            NaN          NaN       NaN             1.06
    2  CDS11 non raw   42.6          72.02       1208.0    0.0035             0.82
    3  CDS11 non 1:4   63.1          72.02       1210.0    0.0075             1.22
    4 CDS11 IPTG raw   78.3          72.02       1212.0    0.0071             1.51
    5 CDS11 IPTG 1:4   80.5          72.02       1215.0    0.0076             1.55
    6  CDS12 non raw   92.7          41.84       1744.0    0.0181             1.79
    7  CDS12 non 1:4   88.1          41.84       1746.0    0.0176             1.70
    8 CDS12 IPTG raw   76.2          41.84       1748.0    0.0128             1.47
    9 CDS12 IPTG 1:4   63.3          41.84       1751.0    0.0386             1.22
   10            P.L   48.5            NaN          NaN       NaN             0.94

=== e40-e281.jpeg ===


 lane        sample  total  predicted_kDa  y_predicted  contrast  total_vs_ladder
    1           P.L   61.8            NaN          NaN       NaN             1.12
    2   e40 non raw   19.8          42.20       1609.0    0.0046             0.36
    3   e40 non 1:4   22.4          42.20       1608.0    0.0045             0.41
    4  e40 IPTG raw   22.5          42.20       1608.0    0.0048             0.41
    5  e40 IPTG 1:4   20.9          42.20       1607.0    0.0021             0.38
    6  e281 non raw   23.9          42.27       1605.0    0.0177             0.43
    7  e281 non 1:4   22.8          42.27       1605.0    0.0216             0.41
    8 e281 IPTG raw   25.9          42.27       1604.0    0.0275             0.47
    9 e281 IPTG 1:4   21.3          42.27       1603.0    0.0052             0.39
   10           P.L   48.7            NaN          NaN       NaN             0.88


### Reading those numbers

**Gel 2.** Total signal says CDS11 lanes carry 0.8–1.5× the ladder lane and CDS12 lanes 1.5–1.8×,
but that is drift in the flattened background, not protein: the CDS11 lane *profiles* have no
discrete peaks anywhere, and the annotated figure shows the lanes as visibly empty apart from a
faint ~15 kDa band. Treat CDS11 lanes 2–5 as **not loaded**.

**Gel 2, CDS12.** Contrast at 41.8 kDa: non 0.0180 / 0.0176, IPTG 0.0128 / 0.0385. The IPTG·raw
lane is below both uninduced lanes. No induction. The most prominent feature in these lanes is a
band at **~50–52 kDa apparent**, present at similar strength in uninduced and induced lanes —
i.e. a host protein, unless leaky expression plus a 20 % anomalous migration is invoked, which the
missing empty-vector control cannot exclude.

**Gel 1 is grossly underloaded** relative to gel 2 — total signal 20–26 vs 43–93. e40 lanes are
blank (contrast 0.002–0.005, the noise floor). e281 lanes 6–8 have faint bands, but at 45–58 kDa,
**above** the 42.3 kDa line and in the same window as gel 2's host bands. Lane 9 (e281 IPTG 1:4)
is the weakest lane on the gel at 0.39×, **below its own uninduced control**.

### Why `1:4` lanes often look crisper than `raw`

Undiluted lysate overloads the well: the lane smears, the local baseline rises and discrete peaks
flatten. So a 1:4 lane can show *higher band contrast* while carrying less protein. This is why
contrast is compared only between lanes of the same dilution, and it is consistent with the
transcribed raw/1:4 order being correct.

## 7. Annotated figures

Eight files, JPEG q92 at dpi 120 (~2150 px wide, ~1 MB each — PNG was 4–8 MB per file because a
noisy photograph does not compress): each gel × {`uniform`, `flattened`} × {`gray`, `color`}.

- **`uniform`** — one linear contrast stretch across the whole frame, nothing else. This is the
  publication-safe version under iGEM's image rules; use it on the wiki.
- **`flattened`** — background-divided so the faint lanes are visible. A non-uniform correction,
  disclosed in the caption. Use it to read the gel, not to publish it.
- **`gray`** — luminance only. Cleanest for band position.
- **`color`** — RGB kept, white-balanced on the background median, saturation lifted globally.
  Coomassie bands come out blue, and the ladder's orange 70 kDa and green 10 kDa reference bands
  show as themselves — a direct visual check that the rungs are on the right bands. The faint
  e281 lanes on gel 1 also read better in colour than in grey.

The **red box** replaces the earlier dashed line. It spans the construct's four lanes, and its
height is the ±9 % calibration precision from §4 — so it is an honest "the band should be inside
this" region, not a false-precision line. It is drawn as a polygon following the gel's tilt.

In [8]:
from matplotlib.patches import Polygon

REF  = {70: "#E8590C", 10: "#2F9E44"}                    # the two coloured reference bands
ORANGE, RED, INK, GREY = "#C2410C", "#DC2626", "#111111", "#6B7280"
SAT  = 1.9        # global saturation multiplier for the colour set
TOL  = 1.09       # the +/-9% calibration precision, used as the box height
COND = ["non","non","IPTG","IPTG"]; DIL = ["raw","1:4","raw","1:4"]

G = {"cds11-cds12.jpeg": dict(gel="Gel 2", xL=196, xR=2496,
        blocks=[("CDS11_WT", 72.02, "#1D4ED8"), ("CDS12_WT", 41.84, "#7C3AED")]),
     "e40-e281.jpeg":    dict(gel="Gel 1", xL=225, xR=2760,
        blocks=[("ANC_Node40", 42.20, "#1D4ED8"), ("ANC_Node281", 42.27, "#7C3AED")])}


def gelimg(fn, corr, palette):
    """corr 'flattened' divides out the illumination, 'uniform' is a plain linear stretch.
       palette 'gray' collapses to luminance, 'color' keeps RGB and lifts saturation."""
    if palette == "gray":
        img = flatfield(fn) if corr == "flattened" else uniform(fn)
        lo, hi = np.percentile(img, [0.3, 99.7] if corr == "flattened" else [2, 98])
        return np.clip((img - lo)/(hi - lo), 0, 1)

    x0, x1, y0, y1 = BOX[fn]
    rgb = load(fn)[y0:y1, x0:x1]
    out = np.empty_like(rgb)
    for k in range(3):
        ch = rgb[..., k]
        out[..., k] = ch/np.clip(gaussian_filter(ch, 70), 1e-3, None) if corr == "flattened" else ch
    # White-balance: the frame is mostly empty gel, so the per-channel medians *are* the
    # background. Dividing by them neutralises the photo's yellow cast; without this the
    # saturation lift below turns the whole gel orange.
    out = out/np.clip(np.median(out.reshape(-1, 3), 0), 1e-3, None)
    lo, hi = np.percentile(out, [0.5, 99.6] if corr == "flattened" else [1, 99])
    out = np.clip((out - lo)/(hi - lo), 0, 1)
    m = out.mean(2, keepdims=True)            # global saturation lift, identical for every pixel
    return np.clip(m + (out - m)*SAT, 0, 1)


def draw(fn, corr, palette):
    c = G[fn]; v = gelimg(fn, corr, palette); H, W = v.shape[:2]
    xL, xR = c["xL"], c["xR"]; lx = np.linspace(xL, xR, 10); step = (xR - xL)/9
    yL = np.array(CAL[fn]["yL"], float); tilt = CAL[fn]["tilt"]
    y_of = lambda mw, f: float(np.polyval(np.polyfit(np.log10(LADDER_MW), yL + f*tilt, 3),
                                          np.log10(mw)))
    HEAD, SIDE = 0.36*H, 0.115*W
    gray = palette == "gray"

    fig = plt.figure(figsize=(16, 16*(H + HEAD)/(W + 2*SIDE)))
    ax  = fig.add_axes([0, 0.042, 1, 0.888])
    ax.imshow(v, cmap="gray" if gray else None, vmin=0 if gray else None,
              vmax=1 if gray else None, aspect="equal", zorder=1)

    # --- ladder: number outside the frame, tick, thin connector to the lane ---
    for f, sgn in [(0., -1), (1., 1)]:
        xtxt   = -0.070*W if sgn < 0 else W + 0.070*W
        t0, t1 = (-0.050*W, -0.012*W) if sgn < 0 else (W + 0.012*W, W + 0.050*W)
        lane_x = xL if sgn < 0 else xR
        for m, y in zip(LADDER_MW, yL + f*tilt):
            col = REF.get(int(m), ORANGE); big = int(m) in REF
            ax.plot([t0, t1], [y, y], color=col, lw=3.4 if big else 2.4,
                    solid_capstyle="butt", clip_on=False, zorder=6)
            ax.plot([t1, lane_x - sgn*step*0.30], [y, y], color=col, lw=1.1, alpha=.50,
                    clip_on=False, zorder=5)
            ax.text(xtxt, y, f"{m:.0f}", color=col, fontsize=17 if big else 15,
                    fontweight="bold", ha="center", va="center", clip_on=False, zorder=7)
        ax.text(xtxt, (yL + f*tilt)[0] - 0.055*H, "kDa", color=ORANGE, fontsize=14,
                style="italic", fontweight="bold", ha="center", va="center",
                clip_on=False, zorder=7)

    # --- header: construct / condition / dilution / lane badge ---
    ax.add_patch(plt.Rectangle((-SIDE, -HEAD), W + 2*SIDE, HEAD, fc="white", ec="none", zorder=2))
    Y = dict(name=-.300, pred=-.252, brk=-.216, cond=-.158, cbrk=-.131, dil=-.083, num=-.034)
    for b, (name, mw, col) in enumerate(c["blocks"]):
        i0, i1 = 1 + 4*b, 4 + 4*b
        a, z = lx[i0] - step*.44, lx[i1] + step*.44
        ax.plot([a, a, z, z], [Y["brk"]*H + .020*H, Y["brk"]*H, Y["brk"]*H, Y["brk"]*H + .020*H],
                color=col, lw=3.0, zorder=6)
        ax.text((a + z)/2, Y["name"]*H, name, color=col, fontsize=23, fontweight="bold",
                ha="center", va="center", zorder=6)
        ax.text((a + z)/2, Y["pred"]*H, f"predicted {mw:.1f} kDa", color=RED, fontsize=16,
                fontweight="bold", ha="center", va="center", zorder=6)
        for k in (0, 2):
            a2, z2 = lx[i0 + k] - step*.40, lx[i0 + k + 1] + step*.40
            ax.plot([a2, a2, z2, z2], [Y["cbrk"]*H + .016*H, Y["cbrk"]*H, Y["cbrk"]*H,
                                       Y["cbrk"]*H + .016*H], color=INK, lw=2.0, zorder=6)
            ax.text((a2 + z2)/2, Y["cond"]*H, COND[k], color=INK, fontsize=17,
                    fontweight="bold", ha="center", va="center", zorder=6)
        for k in range(4):
            ax.text(lx[i0 + k], Y["dil"]*H, DIL[k], color="#374151", fontsize=15,
                    ha="center", va="center", zorder=6)
    for i, x in enumerate(lx):
        pl = i in (0, 9)
        if pl:
            ax.text(x, Y["dil"]*H, "P.L", color="#374151", fontsize=15.5, fontweight="bold",
                    ha="center", va="center", zorder=6)
        ax.add_patch(plt.Circle((x, Y["num"]*H), step*.185, fc=GREY if pl else "#111111",
                                ec="none", zorder=6))
        ax.text(x, Y["num"]*H, str(i + 1), color="white", fontsize=15, fontweight="bold",
                ha="center", va="center", zorder=7)
    ax.text(-SIDE*.94, Y["num"]*H, "lane", color=INK, fontsize=13, fontweight="bold",
            ha="left", va="center", zorder=6)

    # --- predicted-mass BOX: spans the block's four lanes, height = the +/-9% calibration
    #     window, drawn as a polygon so it follows the gel's tilt rather than being flat ---
    for b, (name, mw, col) in enumerate(c["blocks"]):
        i0, i1 = 1 + 4*b, 4 + 4*b
        x0b, x1b = lx[i0] - step*.46, lx[i1] + step*.46
        xs  = np.linspace(x0b, x1b, 40)
        top = [y_of(mw*TOL, (x - xL)/(xR - xL)) for x in xs]
        bot = [y_of(mw/TOL, (x - xL)/(xR - xL)) for x in xs]
        ring = np.c_[np.r_[xs, xs[::-1]], np.r_[top, bot[::-1]]]
        ax.add_patch(Polygon(ring, closed=True, fc=RED, alpha=0.10, ec="none", zorder=4))
        ax.add_patch(Polygon(ring, closed=True, fc="none", ec=RED, lw=3.2,
                             joinstyle="miter", zorder=5))
        ax.text(x1b + step*.10, (top[-1] + bot[-1])/2, f"{mw:.1f}\n±9%", color=RED,
                fontsize=13, fontweight="bold", ha="left", va="center", linespacing=1.25,
                zorder=6, bbox=dict(fc="white", ec=RED, lw=1.5, boxstyle="round,pad=0.26"))

    ax.set_xlim(-SIDE, W + SIDE); ax.set_ylim(H, -HEAD); ax.axis("off")
    fig.text(.5, .985, f"{c['gel']}  ·  2026-09-15 SDS-PAGE, Coomassie  ·  ladder: PageRuler "
             f"Prestained, 170–10 kDa (70 kDa orange, 10 kDa green)",
             fontsize=15, fontweight="bold", ha="center", va="top")
    fig.text(.5, .952, "red box = where that construct's band should fall, widened by the ±9 % "
             "ladder calibration error   ·   lane numbers are position in the loaded series "
             "(the label sheet numbers these 2–11)",
             fontsize=12, color=GREY, ha="center", va="top")
    foot = ("background-flattened for visibility: each channel divided by a 70 px Gaussian estimate "
            "of the transilluminator illumination" if corr == "flattened" else
            "uniform linear contrast stretch only — no background correction")
    if palette == "color":
        foot += f"; white-balanced on the background median, saturation lifted {SAT:.1f}x globally"
    foot += ". No local or per-lane edits."
    fig.text(.5, .006, foot, fontsize=11.5, color=GREY, ha="center", va="bottom", style="italic")

    p = OUT/f"{fn.replace('.jpeg','')}_annotated_{corr}_{palette}.jpg"
    # JPEG: these are photographs with an overlay, and PNG cannot compress a noisy photo
    # (8 MB each at dpi 170). q92 at dpi 120 keeps the text crisp at ~1 MB.
    fig.savefig(p, dpi=120, bbox_inches="tight", facecolor="white", pil_kwargs={"quality": 92}); plt.close(fig)
    return p


for stale in OUT.glob("*_annotated_*.png"):
    stale.unlink()                        # superseded: PNGs were 4-8 MB each
for fn in BOX:
    for corr in ("uniform", "flattened"):
        for palette in ("gray", "color"):
            print("wrote", draw(fn, corr, palette).relative_to(ROOT))

wrote code/output/0915_gel_annotation/cds11-cds12_annotated_uniform_gray.jpg


wrote code/output/0915_gel_annotation/cds11-cds12_annotated_uniform_color.jpg


wrote code/output/0915_gel_annotation/cds11-cds12_annotated_flattened_gray.jpg


wrote code/output/0915_gel_annotation/cds11-cds12_annotated_flattened_color.jpg


wrote code/output/0915_gel_annotation/e40-e281_annotated_uniform_gray.jpg


wrote code/output/0915_gel_annotation/e40-e281_annotated_uniform_color.jpg


wrote code/output/0915_gel_annotation/e40-e281_annotated_flattened_gray.jpg


wrote code/output/0915_gel_annotation/e40-e281_annotated_flattened_color.jpg


## 8. Save the measurements

In [9]:
tables = {fn: analyse(fn) for fn in BOX}
for fn, df in tables.items():
    p = OUT/f"{fn.replace('.jpeg','')}_lane_quant.tsv"
    df.to_csv(p, sep="\t", index=False); print(p.relative_to(ROOT))

cal = pd.DataFrame([dict(gel=fn[:9], band=i+1, mw_kDa=m, y_left=y,
                         y_right=y+CAL[fn]["tilt"])
                    for fn in BOX for i,(m,y) in enumerate(zip(LADDER_MW, CAL[fn]["yL"]))])
p = OUT/"ladder_calibration.tsv"; cal.to_csv(p, sep="\t", index=False); print(p.relative_to(ROOT))

code/output/0915_gel_annotation/cds11-cds12_lane_quant.tsv
code/output/0915_gel_annotation/e40-e281_lane_quant.tsv
code/output/0915_gel_annotation/ladder_calibration.tsv


---
*Log — 2026-09-15: first pass on the two 2026-09-15 expression gels. Ladder identified from the image
alone (nothing in the repo records it) as 10 bands with an orange 4th and a green 10th, then
**confirmed against the PageRuler Prestained product sheet the same day** — top band 170 kDa, not the
180 first assumed. Calibration validated by residual agreement across three independent ladder lanes
(±9 %). Verdict: no detectable induction on either gel; four of the eight sample lanes are
effectively unloaded (CDS11 on gel 2, e40 on gel 1). Three open items: (1) the label sheet numbers
the loaded wells 2–11 but the gel leaves no room for wells outside the two ladders, so the comb size
needs confirming — the construct-to-lane mapping is unaffected either way (§5); (2) the left–right
orientation of both gels is assumed, since both outer lanes are ladder — if a gel was flipped, the
two construct blocks swap; (3) no empty-vector/untransformed-BL21 control was run, so a band present
in both uninduced and induced lanes cannot be assigned to the construct. Next step recommended:
Ni-NTA pulldown or anti-His western, both available without new cloning because every construct
carries pET-28a's N-terminal His6 (see `0910_construct_protein_mw.ipynb`).*

*Log — 2026-09-15 (later): figure labelling reworked on request — larger type, saturated colours,
construct/condition/dilution grouped as brackets above numbered lane badges, ladder rungs labelled
on both sides with the 70 kDa rung orange and the 10 kDa rung green to match the real dye colours.*

*Log — 2026-09-15 (later still): dashed predicted-mass line replaced by a box whose height is the ±9 % calibration window; a colour set added (white-balanced, saturation lifted globally) alongside the grey one. Output naming changed to `_{uniform|flattened}_{gray|color}`.*